# Stage 17A: provenance-safe public strong-base replay

Stage 16B v003の固定manifestへ、Ravaghi public GroupKFold OOFをtarget-safeなcutだけreplayします。提出・再学習・GPUは不要です。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
data_dir = drive_root / 'data'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
assert (data_dir / 'train').is_dir(), f'Missing train data: {data_dir / "train"}'
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

## 入力artifactの自動検出

Stage 16B v003と、以前作成したStage 7の`base_oof.parquet`を使います。Stage 7 artifactがなければNotebook 100を先に実行してください。

In [ ]:
stage16b_run = artifact_dir / 'stage16b_testlike_validation_full_v003'
assert (stage16b_run / 'pseudo_test_manifest.parquet').is_file(), 'Run Notebook 360 Stage 16B v003 first'
preferred_public = artifact_dir / 'stage7_public_residual_gate_full_v001'
public_candidates = sorted({p.parent for p in artifact_dir.glob('**/base_oof.parquet')})
public_oof_run = preferred_public if (preferred_public / 'base_oof.parquet').is_file() else (public_candidates[-1] if public_candidates else None)
assert public_oof_run is not None, 'No base_oof.parquet found. Run notebooks/100_colab_public_residual_gate.ipynb first'
print('Stage 16B:', stage16b_run)
print('Public OOF:', public_oof_run)

## 8坑井smoke

データ整列、target一致、OOF fold provenance、Parquet streamingを確認します。

In [ ]:
SMOKE_ID = 'stage17_public_replay_smoke_v001'
smoke_dir = artifact_dir / SMOKE_ID
if not (smoke_dir / 'summary.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-public-replay',
        '--config', 'configs/experiment/stage17_public_replay.yaml',
        '--stage16b-run', str(stage16b_run), '--public-oof-run', str(public_oof_run),
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir),
        '--run-id', SMOKE_ID, '--limit-wells', '8',
    ], cwd=repo_dir, check=True)
smoke = json.loads((smoke_dir / 'summary.json').read_text())
assert smoke['stage17a_complete'] and smoke['gates']['target_alignment']
smoke

## 773坑井full replay

CPUで実行します。公開OOFを再学習せず読み込み、target-safeな行だけ`replay_predictions.parquet`へstream保存します。

In [ ]:
RUN_ID = 'stage17_public_replay_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'summary.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-public-replay',
        '--config', 'configs/experiment/stage17_public_replay.yaml',
        '--stage16b-run', str(stage16b_run), '--public-oof-run', str(public_oof_run),
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
    ], cwd=repo_dir, check=True)
else:
    print('Reusing completed run:', run_dir)
summary = json.loads((run_dir / 'summary.json').read_text())
{
    'stage17a_complete': summary['stage17a_complete'],
    'promoted_to_selector_replay': summary['promoted_to_selector_replay'],
    'source_branch': summary['source_branch'],
    'n_wells': summary['n_wells'], 'n_cuts': summary['n_cuts'],
    'maximum_target_alignment_difference': summary['maximum_target_alignment_difference'],
    'primary': summary['role_report']['primary'],
    'diagnostic': summary['role_report']['diagnostic'],
    'improved_eligible_folds': summary['improved_eligible_folds'],
    'improved_hybrid_folds': summary['improved_hybrid_folds'],
    'gates': summary['gates'],
    'unvalidated_v599_components': summary['unvalidated_v599_components'],
    'next_step': summary['next_step'],
}

最後の辞書を共有してください。ここではKaggle提出は行いません。通過した場合は、未カバー短prefixへSP45 selectorをreplayするStage 17Bへ進みます。